# Clustering Algorithms

A comprehensive guide to clustering: K-Means, Hierarchical, and DBSCAN.

## Learning Objectives

- Understand clustering as unsupervised learning
- Master K-Means algorithm and implementation
- Learn Hierarchical clustering and dendrograms
- Implement DBSCAN for density-based clustering
- Evaluate cluster quality with internal metrics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Introduction to Clustering

**Clustering** is an unsupervised learning technique that groups similar data points together.

**Key Differences from Classification:**
- No labeled data (we don't know the true groups)
- Discover hidden structure in data
- Number of clusters may be unknown

**Common Applications:**
- Customer segmentation
- Image segmentation
- Anomaly detection
- Document grouping

In [ ]:
# Generate sample data for demonstration
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

# Visualize the datasets
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', s=30)
axes[0].set_title('Blob Clusters (Ideal for K-Means)')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', s=30)
axes[1].set_title('Moon Shapes (Non-convex)')

axes[2].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', s=30)
axes[2].set_title('Concentric Circles')

plt.tight_layout()
plt.show()

## 2. K-Means Clustering

**Algorithm:**
1. Initialize k centroids randomly
2. Assign each point to nearest centroid
3. Update centroids as mean of assigned points
4. Repeat until convergence

**Objective:** Minimize within-cluster sum of squares (inertia):
$$\text{WCSS} = \sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2$$

In [ ]:
# K-Means from scratch
class KMeansFromScratch:
    def __init__(self, n_clusters=3, max_iters=100, random_state=42):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        
    def fit(self, X):
        np.random.seed(self.random_state)
        
        # Initialize centroids randomly from data points
        idx = np.random.choice(len(X), self.n_clusters, replace=False)
        self.centroids = X[idx].copy()
        
        for _ in range(self.max_iters):
            # Assign labels based on closest centroid
            distances = self._compute_distances(X)
            self.labels = np.argmin(distances, axis=1)
            
            # Update centroids
            new_centroids = np.array([X[self.labels == k].mean(axis=0) 
                                       for k in range(self.n_clusters)])
            
            # Check convergence
            if np.allclose(self.centroids, new_centroids):
                break
            self.centroids = new_centroids
            
        return self
    
    def _compute_distances(self, X):
        distances = np.zeros((len(X), self.n_clusters))
        for k, centroid in enumerate(self.centroids):
            distances[:, k] = np.linalg.norm(X - centroid, axis=1)
        return distances
    
    def predict(self, X):
        distances = self._compute_distances(X)
        return np.argmin(distances, axis=1)

# Compare with sklearn
kmeans_scratch = KMeansFromScratch(n_clusters=4)
kmeans_scratch.fit(X_blobs)

kmeans_sklearn = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_sklearn.fit(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans_scratch.labels, cmap='viridis', s=30)
axes[0].scatter(kmeans_scratch.centroids[:, 0], kmeans_scratch.centroids[:, 1], 
                c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[0].set_title('K-Means from Scratch')

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=kmeans_sklearn.labels_, cmap='viridis', s=30)
axes[1].scatter(kmeans_sklearn.cluster_centers_[:, 0], kmeans_sklearn.cluster_centers_[:, 1], 
                c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[1].set_title('Scikit-learn K-Means')

plt.tight_layout()
plt.show()

## 3. Choosing the Number of Clusters (k)

Two popular methods:
1. **Elbow Method**: Look for "elbow" in inertia plot
2. **Silhouette Score**: Maximize average silhouette

In [ ]:
# Elbow method and Silhouette analysis
k_range = range(2, 11)
inertias = []
silhouettes = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_blobs)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_blobs, kmeans.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow plot
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')
axes[0].axvline(x=4, color='r', linestyle='--', label='Optimal k=4')
axes[0].legend()

# Silhouette plot
axes[1].plot(k_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].axvline(x=4, color='r', linestyle='--', label='Optimal k=4')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Best k by Silhouette: {k_range[np.argmax(silhouettes)]}")

In [ ]:
# Silhouette plot visualization
from matplotlib import cm

def plot_silhouette(X, labels, ax):
    n_clusters = len(np.unique(labels))
    silhouette_avg = silhouette_score(X, labels)
    sample_silhouette_values = silhouette_samples(X, labels)
    
    y_lower = 10
    for i in range(n_clusters):
        cluster_silhouette = sample_silhouette_values[labels == i]
        cluster_silhouette.sort()
        
        size = cluster_silhouette.shape[0]
        y_upper = y_lower + size
        
        color = cm.nipy_spectral(float(i) / n_clusters)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette, 
                         facecolor=color, edgecolor=color, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * size, str(i))
        y_lower = y_upper + 10
    
    ax.axvline(x=silhouette_avg, color='red', linestyle='--', label=f'Avg: {silhouette_avg:.2f}')
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster')
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, k in zip(axes, [3, 4, 5]):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_blobs)
    plot_silhouette(X_blobs, labels, ax)
    ax.set_title(f'k={k}')

plt.tight_layout()
plt.show()

## 4. K-Means Limitations

In [ ]:
# K-Means fails on non-convex clusters
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

datasets = [('Moons', X_moons), ('Circles', X_circles)]

for row, (name, X) in enumerate(datasets):
    # K-Means
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    labels_km = kmeans.fit_predict(X)
    
    # DBSCAN
    dbscan = DBSCAN(eps=0.3, min_samples=5)
    labels_db = dbscan.fit_predict(X)
    
    # Spectral Clustering
    from sklearn.cluster import SpectralClustering
    spectral = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', random_state=42)
    labels_sp = spectral.fit_predict(X)
    
    axes[row, 0].scatter(X[:, 0], X[:, 1], c=labels_km, cmap='viridis', s=30)
    axes[row, 0].set_title(f'{name}: K-Means')
    
    axes[row, 1].scatter(X[:, 0], X[:, 1], c=labels_db, cmap='viridis', s=30)
    axes[row, 1].set_title(f'{name}: DBSCAN')
    
    axes[row, 2].scatter(X[:, 0], X[:, 1], c=labels_sp, cmap='viridis', s=30)
    axes[row, 2].set_title(f'{name}: Spectral')

plt.tight_layout()
plt.show()

## 5. Hierarchical Clustering

**Types:**
- **Agglomerative** (bottom-up): Start with individual points, merge
- **Divisive** (top-down): Start with all points, split

**Linkage Methods:**
- `single`: Min distance between clusters
- `complete`: Max distance between clusters
- `average`: Average distance between clusters
- `ward`: Minimize variance increase (similar to K-Means)

In [ ]:
# Create dendrogram
from scipy.cluster.hierarchy import dendrogram, linkage

# Use a smaller sample for visualization
X_sample = X_blobs[:50]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

linkage_methods = ['single', 'complete', 'average', 'ward']

for ax, method in zip(axes.ravel(), linkage_methods):
    Z = linkage(X_sample, method=method)
    dendrogram(Z, ax=ax, leaf_rotation=90)
    ax.set_title(f'{method.capitalize()} Linkage')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Distance')

plt.tight_layout()
plt.show()

In [ ]:
# Agglomerative clustering with different linkages
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, method in zip(axes.ravel(), linkage_methods):
    agg = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels = agg.fit_predict(X_blobs)
    
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='viridis', s=30)
    ax.set_title(f'{method.capitalize()} Linkage\nSilhouette: {silhouette_score(X_blobs, labels):.3f}')

plt.tight_layout()
plt.show()

## 6. DBSCAN (Density-Based Clustering)

**Key Parameters:**
- `eps`: Maximum distance between two samples in same neighborhood
- `min_samples`: Minimum points required to form a dense region

**Point Types:**
- **Core Points**: Have ≥ min_samples within eps
- **Border Points**: Within eps of a core point
- **Noise Points**: Neither core nor border (labeled -1)

In [ ]:
# DBSCAN on moons dataset
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

eps_values = [0.1, 0.2, 0.3]
min_samples_values = [3, 5, 10]

for i, eps in enumerate(eps_values):
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_moons)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    axes[0, i].scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='viridis', s=30)
    axes[0, i].set_title(f'eps={eps}\nClusters: {n_clusters}, Noise: {n_noise}')

for i, min_samp in enumerate(min_samples_values):
    dbscan = DBSCAN(eps=0.2, min_samples=min_samp)
    labels = dbscan.fit_predict(X_moons)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    axes[1, i].scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='viridis', s=30)
    axes[1, i].set_title(f'min_samples={min_samp}\nClusters: {n_clusters}, Noise: {n_noise}')

plt.suptitle('DBSCAN Parameter Effects', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Finding optimal eps using k-distance graph
from sklearn.neighbors import NearestNeighbors

def plot_k_distance(X, k=5):
    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors.fit(X)
    distances, _ = neighbors.kneighbors(X)
    
    # Sort distances to k-th neighbor
    k_distances = np.sort(distances[:, k-1])
    
    plt.figure(figsize=(10, 5))
    plt.plot(k_distances)
    plt.xlabel('Points sorted by distance')
    plt.ylabel(f'Distance to {k}-th nearest neighbor')
    plt.title(f'K-Distance Graph (k={k})\nLook for "elbow" to find optimal eps')
    plt.grid(True)
    plt.show()

plot_k_distance(X_moons, k=5)

## 7. Cluster Evaluation Metrics

**Internal Metrics (no ground truth):**
- **Silhouette Score**: Measures cohesion vs separation (-1 to 1, higher is better)
- **Calinski-Harabasz Index**: Ratio of between-cluster to within-cluster variance
- **Davies-Bouldin Index**: Average similarity of clusters (lower is better)

In [ ]:
# Compare algorithms with evaluation metrics
algorithms = {
    'K-Means': KMeans(n_clusters=4, random_state=42, n_init=10),
    'Agglomerative (Ward)': AgglomerativeClustering(n_clusters=4, linkage='ward'),
    'Agglomerative (Complete)': AgglomerativeClustering(n_clusters=4, linkage='complete'),
    'DBSCAN': DBSCAN(eps=0.5, min_samples=5)
}

results = []
for name, algo in algorithms.items():
    labels = algo.fit_predict(X_blobs)
    
    # DBSCAN may have noise points (-1), filter them for metrics
    mask = labels != -1
    if mask.sum() < 2 or len(set(labels[mask])) < 2:
        continue
        
    results.append({
        'Algorithm': name,
        'Silhouette': silhouette_score(X_blobs[mask], labels[mask]),
        'Calinski-Harabasz': calinski_harabasz_score(X_blobs[mask], labels[mask]),
        'Davies-Bouldin': davies_bouldin_score(X_blobs[mask], labels[mask]),
        'Clusters': len(set(labels[mask]))
    })

results_df = pd.DataFrame(results)
print("Clustering Algorithm Comparison:")
print(results_df.to_string(index=False))

## 8. Practical Example: Customer Segmentation

In [ ]:
# Simulate customer data
np.random.seed(42)
n_customers = 500

# Create customer segments
customers = pd.DataFrame({
    'age': np.concatenate([
        np.random.normal(25, 5, 150),  # Young
        np.random.normal(45, 8, 200),  # Middle-aged
        np.random.normal(65, 7, 150)   # Senior
    ]),
    'annual_income': np.concatenate([
        np.random.normal(30000, 8000, 150),   # Low income
        np.random.normal(75000, 15000, 200),  # Medium income
        np.random.normal(50000, 10000, 150)   # Moderate income
    ]),
    'spending_score': np.concatenate([
        np.random.normal(70, 15, 150),  # High spenders
        np.random.normal(50, 20, 200),  # Medium spenders
        np.random.normal(30, 10, 150)   # Low spenders
    ])
})

# Clip to valid ranges
customers['age'] = customers['age'].clip(18, 85)
customers['annual_income'] = customers['annual_income'].clip(10000, 150000)
customers['spending_score'] = customers['spending_score'].clip(1, 100)

print(customers.describe())

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customers)

# Find optimal k
k_range = range(2, 9)
silhouettes = [silhouette_score(X_scaled, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)) 
               for k in k_range]

plt.figure(figsize=(8, 4))
plt.plot(k_range, silhouettes, 'go-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Optimal Number of Customer Segments')
plt.show()

optimal_k = k_range[np.argmax(silhouettes)]
print(f"Optimal number of segments: {optimal_k}")

In [ ]:
# Cluster customers
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
customers['segment'] = kmeans.fit_predict(X_scaled)

# Analyze segments
segment_summary = customers.groupby('segment').agg({
    'age': ['mean', 'std'],
    'annual_income': ['mean', 'std'],
    'spending_score': ['mean', 'std']
}).round(2)

print("\nCustomer Segment Profiles:")
print(segment_summary)

# Visualize segments
fig = plt.figure(figsize=(12, 5))

# 2D view
ax1 = fig.add_subplot(121)
scatter = ax1.scatter(customers['annual_income'], customers['spending_score'], 
                      c=customers['segment'], cmap='viridis', s=30)
ax1.set_xlabel('Annual Income')
ax1.set_ylabel('Spending Score')
ax1.set_title('Customer Segments')
plt.colorbar(scatter, ax=ax1, label='Segment')

# 3D view
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(customers['age'], customers['annual_income'], customers['spending_score'],
            c=customers['segment'], cmap='viridis', s=30)
ax2.set_xlabel('Age')
ax2.set_ylabel('Income')
ax2.set_zlabel('Spending')
ax2.set_title('3D Customer Segments')

plt.tight_layout()
plt.show()

## 9. Algorithm Comparison Summary

| Algorithm | Strengths | Weaknesses | Best For |
|-----------|-----------|------------|----------|
| K-Means | Fast, simple | Assumes spherical, needs k | Large datasets, convex clusters |
| Hierarchical | Dendrogram, no k needed | O(n³), sensitive to noise | Small data, hierarchy exploration |
| DBSCAN | Finds arbitrary shapes, detects noise | Sensitive to eps, min_samples | Varying density, outlier detection |

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Algorithm': ['K-Means', 'Hierarchical', 'DBSCAN'],
    'Complexity': ['O(n*k*i)', 'O(n³)', 'O(n log n)'],
    'Cluster Shape': ['Spherical', 'Any', 'Arbitrary'],
    'Outliers': ['No handling', 'Sensitive', 'Detects'],
    'Scalability': ['Excellent', 'Poor', 'Good'],
    'Parameters': ['k', 'k or cut height', 'eps, min_samples']
})

print("Clustering Algorithm Comparison:")
print(summary.to_string(index=False))

## 10. Key Takeaways

1. **K-Means** is fast and works well for spherical, well-separated clusters
2. Use **Elbow Method** and **Silhouette Score** to find optimal k
3. **Hierarchical clustering** provides dendrograms for exploratory analysis
4. **DBSCAN** handles arbitrary shapes and automatically detects noise
5. **Always scale features** before clustering
6. **No single best algorithm** - choose based on data characteristics
7. Validate clusters with **internal metrics** and **domain knowledge**